# ESAVE Validation Harness

**Purpose**: Run ESAVE validation notebooks and view results — all inside JupyterHub.  
**How it works**: This harness imports `notebook_bridge`, executes the validation template with your parameters, and renders the dashboard inline in this notebook.  
**No server required**: Everything runs locally through nbclient. No Flask, no Gradio, no Streamlit, no HTTP ports.

---

## Step 1: Verify environment

In [ ]:
# Run this cell first to confirm all dependencies are available
from notebook_bridge import check_environment
check_environment()

---
## Step 2: Configure your validation run

Edit the parameters below to match your validation target.

In [ ]:
# ┌─────────────────────────────────────────────────┐
# │  EDIT THESE PARAMETERS FOR YOUR RUN             │
# └─────────────────────────────────────────────────┘

# Path to the ESAVE validation template notebook
TEMPLATE_NOTEBOOK = 'notebooks/rag_validation_pipeline.ipynb'

# Parameters to inject into the template
RUN_PARAMS = {
    'spec_path': 'dpas_hce_ios_spec_v1.0.pdf',
    'embedding_model': 'mixedbread-ai/mxbai-embed-large-v1',
    'quality_threshold': 0.85,
    'chunk_size': 1024,
    'chunk_overlap': 128,
}

# Where to save the executed notebook
EXECUTED_OUTPUT = 'sample_runs/validation_executed.ipynb'

# Optional: also save a static HTML file (Tier 1 output)
# Set to None to skip, or a path to save
ALSO_SAVE_HTML = 'output/dashboard.html'

# Execution settings
KERNEL_NAME = 'python3'
TIMEOUT = 600  # seconds per cell

print('Configuration ready.')
print(f'  Template:  {TEMPLATE_NOTEBOOK}')
print(f'  Params:    {list(RUN_PARAMS.keys())}')
print(f'  Output:    {EXECUTED_OUTPUT}')
print(f'  HTML copy: {ALSO_SAVE_HTML or "(none)"}')

---
## Step 3: Execute and view dashboard

This cell does three things:
1. Injects your parameters into the template notebook
2. Executes all cells through nbclient
3. Renders the dashboard right here in this cell

In [ ]:
from notebook_bridge import run_and_display

executed_path = run_and_display(
    TEMPLATE_NOTEBOOK,
    params=RUN_PARAMS,
    output_path=EXECUTED_OUTPUT,
    kernel_name=KERNEL_NAME,
    timeout=TIMEOUT,
    also_save_html=ALSO_SAVE_HTML,
)

print(f'\nExecuted notebook saved to: {executed_path}')
if ALSO_SAVE_HTML:
    print(f'Static HTML also saved to: {ALSO_SAVE_HTML}')

---
## Alternative: View a previously executed notebook

If you already have an executed notebook and just want to view its dashboard:

In [ ]:
# Uncomment to view an existing executed notebook:

# from notebook_bridge import display_dashboard
# display_dashboard('sample_runs/validation_executed.ipynb')

---
## Alternative: Quick summary only

If you just want key metrics without the full dashboard:

In [ ]:
# Uncomment to view just the summary bar:

# from notebook_bridge import display_summary
# display_summary('sample_runs/validation_executed.ipynb')

---
## Extract structured data programmatically

For downstream processing (e.g., feeding results back into ESAVE agents):

In [ ]:
# Uncomment to extract structured data:

# from notebook_bridge import NotebookParser
# parsed = NotebookParser.parse('sample_runs/validation_executed.ipynb')
# summary = NotebookParser.extract_summary(parsed)
# 
# bridge_data = summary['bridge_data']
# gap_data = summary['gap_data']
# 
# # Access specific fields
# print(f"Pass rate: {bridge_data['pass_count']}/{bridge_data['total_sections']}")
# print(f"Mean score: {bridge_data['mean_score']}")
# print(f"Gaps found: {len(gap_data)}")
# 
# # Feed failures into ESAVE for deeper analysis
# failures = [s for s in bridge_data['validation_data'] if s['status'] == 'FAIL']
# for f in failures:
#     print(f"  FAIL: §{f['section_id']} — {f['requirement_type']} (score: {f['match_score']})")